In [18]:
from dataclasses import dataclass

import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from occhio.autoencoder import TiedLinearRelu
from occhio.distributions.hierarchical import HierarchicalSparse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps")
print(f"Using device: {device}")


Using device: mps


In [19]:
@dataclass
class SweepResult:
    mean_norm: torch.Tensor
    mean_superposition: torch.Tensor
    mean_eval_loss: torch.Tensor
    model_weights: torch.Tensor
    model_biases: torch.Tensor


def make_generator(seed: int) -> torch.Generator:
    generator_device = "mps" # "cuda" if device.type == "cuda" else "cpu"
    generator = torch.Generator(device=generator_device)
    generator.manual_seed(seed)
    return generator


def drop_worst_and_average(values: torch.Tensor, losses: torch.Tensor) -> float:
    if values.numel() <= 1:
        return float(values.mean().item())

    keep_mask = torch.ones_like(losses, dtype=torch.bool)
    worst_index = int(losses.argmax().item())
    keep_mask[worst_index] = False
    return float(values[keep_mask].mean().item())


def train_single_model(
    p_feature1: float,
    p_feature2_conditional: float,
    feature2_importance: float,
    train_steps: int,
    batch_size: int,
    eval_batch_size: int,
    learning_rate: float,
    weight_decay: float,
    seed: int,
    log_loss_every: int = 10,
) -> tuple[float, float, float, torch.Tensor, torch.Tensor]:
    generator = make_generator(seed)

    distribution = HierarchicalSparse(
        n_features=3,
        p_by_depth=[1.0, p_feature1, p_feature2_conditional],
        max_children=1,
        device=device,
        generator=generator,
    )

    model = TiedLinearRelu(
        n_features=2,
        n_hidden=1,
        device=device,
        generator=generator,
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    importances = torch.tensor([1.0, feature2_importance], device=device)

    for epoch in range(train_steps):
        x_full = distribution.sample(batch_size)
        x = x_full[:, 1:3]
        x_hat, _ = model(x)
        loss = model.loss(x, x_hat, importances)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if log_loss_every > 0 and ((epoch + 1) % log_loss_every == 0):
            print(
                f"seed={seed} p1={p_feature1:.3f} p2|1={p_feature2_conditional:.3f} "
                f"imp2={feature2_importance:.3f} epoch={epoch + 1}/{train_steps} "
                f"loss={loss.item():.6f}"
            )

    with torch.no_grad():
        x_eval = distribution.sample(eval_batch_size)[:, 1:3]
        x_hat_eval, _ = model(x_eval)
        eval_loss = float(model.loss(x_eval, x_hat_eval, importances).item())

        weight_vector = model.W.detach().squeeze(0)
        bias_vector = model.b.detach()

        feature1_norm = torch.linalg.vector_norm(weight_vector[0:1], ord=2)
        feature2_norm = torch.linalg.vector_norm(weight_vector[1:2], ord=2)

        # For i=2, superposition is sum_{j != 2} (W_hat_2 · W_j)^2.
        # With two features, this is just j=1.
        unit_feature2 = weight_vector.new_tensor(0.0)
        if feature2_norm > 1e-8:
            unit_feature2 = weight_vector[1] / feature2_norm

        feature2_superposition = float((unit_feature2 * weight_vector[0]).pow(2).item())

    return (
        float(feature2_norm.item()),
        feature2_superposition,
        eval_loss,
        weight_vector.to(device),
        bias_vector.to(device),
    )


def run_phase_sweep_for_p_feature1(
    p_feature1: float,
    importance_axis: torch.Tensor,
    conditional_density_axis: torch.Tensor,
    ensemble_size: int = 10,
    train_steps: int = 800,
    batch_size: int = 256,
    eval_batch_size: int = 4096,
    learning_rate: float = 3e-4,
    weight_decay: float = 1e-4,
) -> SweepResult:
    ny = conditional_density_axis.numel()
    nx = importance_axis.numel()

    mean_norm = torch.zeros(ny, nx)
    mean_superposition = torch.zeros(ny, nx)
    mean_eval_loss = torch.zeros(ny, nx)

    model_weights = torch.zeros(ny, nx, ensemble_size, 2)
    model_biases = torch.zeros(ny, nx, ensemble_size, 2)

    p1_hash = int(round(p_feature1 * 1_000_000))

    for y_index, p_feature2_conditional in enumerate(conditional_density_axis.tolist()):
        for x_index, feature2_importance in enumerate(importance_axis.tolist()):
            norms = []
            supers = []
            losses = []
            weights = []
            biases = []

            for model_index in range(ensemble_size):
                seed = (
                    p1_hash * 1_000_003
                    + y_index * 10_007
                    + x_index * 1_009
                    + model_index
                ) % (2**31 - 1)

                norm_value, superposition_value, loss_value, weight_vector, bias_vector = train_single_model(
                    p_feature1=p_feature1,
                    p_feature2_conditional=p_feature2_conditional,
                    feature2_importance=feature2_importance,
                    train_steps=train_steps,
                    batch_size=batch_size,
                    eval_batch_size=eval_batch_size,
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    seed=seed,
                )

                norms.append(norm_value)
                supers.append(superposition_value)
                losses.append(loss_value)
                weights.append(weight_vector)
                biases.append(bias_vector)

            norm_tensor = torch.tensor(norms)
            super_tensor = torch.tensor(supers)
            loss_tensor = torch.tensor(losses)

            mean_norm[y_index, x_index] = drop_worst_and_average(norm_tensor, loss_tensor)
            mean_superposition[y_index, x_index] = drop_worst_and_average(super_tensor, loss_tensor)
            mean_eval_loss[y_index, x_index] = drop_worst_and_average(loss_tensor, loss_tensor)

            model_weights[y_index, x_index] = torch.stack(weights, dim=0)
            model_biases[y_index, x_index] = torch.stack(biases, dim=0)

    return SweepResult(
        mean_norm=mean_norm,
        mean_superposition=mean_superposition,
        mean_eval_loss=mean_eval_loss,
        model_weights=model_weights,
        model_biases=model_biases,
    )


def phase_rgb(norm: torch.Tensor, superposition: torch.Tensor) -> torch.Tensor:
    norm = norm.clamp(0.0, 1.0)
    superposition = superposition.clamp(0.0, 1.0)

    n = (1.0 - norm).unsqueeze(-1)
    s = superposition.unsqueeze(-1)

    color_top_left = torch.tensor([0.10, 0.13, 0.96])
    color_top_right = torch.tensor([0.97, 0.07, 0.18])
    color_bottom_left = torch.tensor([0.84, 0.85, 0.88])
    color_bottom_right = torch.tensor([0.97, 0.94, 0.95])

    rgb = (
        (1 - n) * (1 - s) * color_top_left
        + (1 - n) * s * color_top_right
        + n * (1 - s) * color_bottom_left
        + n * s * color_bottom_right
    )
    return rgb.clamp(0.0, 1.0)


def rgb_to_css(rgb: torch.Tensor) -> list[str]:
    rgb_255 = (rgb * 255).round().to(torch.int64).view(-1, 3).tolist()
    return [f"rgb({r},{g},{b})" for r, g, b in rgb_255]


In [20]:
p_feature1_values = [0.01, 0.1, 0.5, 0.9, 0.99]



# p_feature1_values = [0.01, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99]
importance_axis = torch.logspace(-1, 1, 20)
conditional_density_axis = torch.logspace(-2, 0, 20)

ensemble_size = 5
train_steps = 200
batch_size = 64
eval_batch_size = 4096
learning_rate = 3e-4
weight_decay = 1e-4

results: dict[float, SweepResult] = {}

for subplot_index, p_feature1 in enumerate(p_feature1_values):
    print(
        f"Training subplot {subplot_index + 1}/{len(p_feature1_values)} at p(feature1)={p_feature1:.2f}"
    )

    result = run_phase_sweep_for_p_feature1(
        p_feature1=p_feature1,
        importance_axis=importance_axis,
        conditional_density_axis=conditional_density_axis,
        ensemble_size=ensemble_size,
        train_steps=train_steps,
        batch_size=batch_size,
        eval_batch_size=eval_batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    )
    results[p_feature1] = result

print("Training complete.")


Training subplot 1/5 at p(feature1)=0.01
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=10/200 loss=0.001251
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=20/200 loss=0.001452
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=30/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=40/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=50/200 loss=0.000128
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=60/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=70/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=80/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=90/200 loss=0.000006
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=100/200 loss=0.001396
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=110/200 loss=0.000000
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=120/200 loss=0.000737
seed=1410095412 p1=0.010 p2|1=0.010 imp2=0.100 epoch=130/200 loss=0.

KeyboardInterrupt: 

In [ ]:
torch.set_printoptions(threshold=1_000_000, linewidth=180, sci_mode=False)

for p_feature1 in p_feature1_values:
    result = results[p_feature1]

    print(f"\n=== p(feature1)={p_feature1:.2f} ===")
    print(f"biases shape: {tuple(result.model_biases.shape)}")
    print(result.model_biases)

    print(f"\n=== p(feature1)={p_feature1:.2f} ===")
    print(f"weights shape: {tuple(result.model_weights.shape)}")
    print(result.model_weights)


In [ ]:
fig = make_subplots(
    rows=3,
    cols=3,
    subplot_titles=[f"p(feature1) = {p:.2f}" for p in p_feature1_values],
    horizontal_spacing=0.04,
    vertical_spacing=0.08,
)

x_grid = importance_axis.unsqueeze(0).repeat(conditional_density_axis.numel(), 1)
y_grid = conditional_density_axis.unsqueeze(1).repeat(1, importance_axis.numel())
tile_size = 7.0

for idx, p_feature1 in enumerate(p_feature1_values):
    row = (idx // 3) + 1
    col = (idx % 3) + 1

    result = results[p_feature1]
    rgb = phase_rgb(result.mean_norm, result.mean_superposition)

    custom_data = torch.stack(
        [
            result.mean_norm.reshape(-1),
            result.mean_superposition.reshape(-1),
            result.mean_eval_loss.reshape(-1),
        ],
        dim=-1,
    ).tolist()

    fig.add_trace(
        go.Scatter(
            x=x_grid.reshape(-1).tolist(),
            y=y_grid.reshape(-1).tolist(),
            mode="markers",
            marker={
                "symbol": "square",
                "size": tile_size,
                "color": rgb_to_css(rgb),
                "line": {"width": 0},
            },
            customdata=custom_data,
            hovertemplate=(
                "importance(feature2)=%{x:.4f}<br>"
                "p(feature2|feature1)=%{y:.4f}<br>"
                "||W2||=%{customdata[0]:.4f}<br>"
                "superposition=%{customdata[1]:.4f}<br>"
                "eval_loss=%{customdata[2]:.6f}<extra></extra>"
            ),
            showlegend=False,
        ),
        row=row,
        col=col,
    )

    fig.update_xaxes(
        type="log",
        tickmode="array",
        tickvals=[0.1, 1.0, 10.0],
        ticktext=["0.1", "1.0", "10"],
        row=row,
        col=col,
    )

    fig.update_yaxes(
        type="log",
        tickmode="array",
        tickvals=[0.01, 0.1, 1.0],
        ticktext=["0.01", "0.1", "1.0"],
        row=row,
        col=col,
    )

for axis_index in range(1, len(p_feature1_values) + 1):
    axis_suffix = "" if axis_index == 1 else str(axis_index)
    fig.layout[f"yaxis{axis_suffix}"].update(
        scaleanchor=f"x{axis_suffix}",
        scaleratio=1,
    )

for col in [1, 2, 3]:
    fig.update_xaxes(title_text="Feature 2 Importance", row=3, col=col)

for row in [1, 2, 3]:
    fig.update_yaxes(title_text="p(feature2 | feature1)", row=row, col=1)

fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

fig.update_layout(
    title="Hierarchical Superposition Phase Diagram (2 features, 1 latent dim)",
    width=1500,
    height=900,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=30, r=15, t=70, b=35),
)

fig.show()


In [ ]:
legend_resolution = 100
legend_norm = torch.linspace(0.0, 1.0, legend_resolution)
legend_super = torch.linspace(0.0, 1.0, legend_resolution)
legend_super_grid, legend_norm_grid = torch.meshgrid(legend_super, legend_norm, indexing="xy")
legend_rgb = phase_rgb(legend_norm_grid, legend_super_grid)

legend_fig = go.Figure(
    go.Scatter(
        x=legend_super_grid.reshape(-1).tolist(),
        y=legend_norm_grid.reshape(-1).tolist(),
        mode="markers",
        marker={
            "symbol": "square",
            "size": 5.2,
            "color": rgb_to_css(legend_rgb),
            "line": {"width": 0},
        },
        hoverinfo="skip",
        showlegend=False,
    )
)

legend_fig.update_xaxes(
    tickmode="array",
    tickvals=[0.0, 0.5, 1.0],
    ticktext=["0", "0.5", "≥1"],
    title="Superposition",
    range=[0.0, 1.0],
    showgrid=False,
    zeroline=False,
)
legend_fig.update_yaxes(
    tickmode="array",
    tickvals=[0.0, 0.5, 1.0],
    ticktext=["0", "0.5", "≥1"],
    title="||W2||",
    range=[0.0, 1.0],
    showgrid=False,
    zeroline=False,
    scaleanchor="x",
    scaleratio=1,
)
legend_fig.update_layout(
    title="Color Map: Norm vs Superposition",
    width=450,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
legend_fig.show()


In [ ]:
# Debug-only diagnostics: run this after the training cell and before plotting changes.
for p_feature1 in p_feature1_values:
    result = results[p_feature1]
    norm_flat = result.mean_norm.reshape(-1)
    super_flat = result.mean_superposition.reshape(-1)

    corr = float(torch.corrcoef(torch.stack([norm_flat, super_flat]))[0, 1])
    red_like = int(((norm_flat > 0.6) & (super_flat > 0.6)).sum())
    blue_like = int(((norm_flat > 0.6) & (super_flat < 0.2)).sum())

    print(
        f"p(feature1)={p_feature1:.2f} | corr(norm,super)={corr:+.3f} | red_like={red_like} | blue_like={blue_like}"
    )
